## Week 2 Day 2

Our first Agentic Framework project!!

Prepare yourself for something ridiculously easy.

We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">IMPORTANT PLEASE READ - Emails with SendGrid</h2>
            <span style="color:#ff7800;">We are going to use the email provider SendGrid. But this is OPTIONAL.<br/>
            The next cell contains instructions for SendGrid. But if this gives you problems, or if you'd rather use any alternative,<br/>
            please see <a href="https://edwarddonner.com/faq">Q29 on the FAQ page here</a> for the full explanation and alternatives.
            </span>
        </td>
    </tr>
</table>

## Setting up SendGrid


Please visit Sendgrid at: https://sendgrid.com/

(Sendgrid is a Twilio company for sending emails.)

__If SendGrid gives you problems, see the alternative implementations on Q29 of my FAQ page at https://edwarddonner.com/faq that includes "Resend Email" in community_contributions/2_lab2_with_resend_email and just skipping email altogether.__

Setting up a SendGrid account is free! (at least, for me, right now).

Once you've created an account, click on:

Settings (left sidebar) >> API Keys >> Create API Key (button on top right)

Copy the key to the clipboard, then add a new line to your .env file:

`SENDGRID_API_KEY=xxxx`

And also, within SendGrid, go to:

Settings (left sidebar) >> Sender Authentication >> "Verify a Single Sender"  
and verify that your own email address is a real email address, so that SendGrid can send emails for you.

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool, add_trace_processor, OpenAIChatCompletionsModel
from agents.tracing.processors import ConsoleSpanExporter, BatchTraceProcessor
from openai import AsyncAzureOpenAI
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
import sendgrid
import os
from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio
import requests


In [2]:
load_dotenv(override=True)

azure_client = AsyncAzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    api_version=os.getenv("AZURE_OPENAI_API_VERSION") or "2024-12-01-preview",
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
)

deployment_name = os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME") or os.getenv("AZURE_OPENAI_DEPLOYMENT")

if not deployment_name:
    raise ValueError("Set AZURE_OPENAI_CHAT_DEPLOYMENT_NAME (or AZURE_OPENAI_DEPLOYMENT) in your .env")


console_exporter = ConsoleSpanExporter()
console_processor = BatchTraceProcessor(console_exporter)
add_trace_processor(console_processor)

In [6]:
# Let's just check emails are working for you

def send_test_html_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body to all sales prospects using Resend"""
    
    from_email = "onboarding@resend.dev"
    to_email = "plspls@dontsp.am"
    RESEND_API_KEY = os.environ.get("RESEND_API_KEY")
    
    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "from": f"Ed Donner <{from_email}>",
        "to": [to_email],
        "subject": subject,
        "html": html_body
    }
    
    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)
    
    if response.status_code == 202:
        return "Email sent successfully"
    else:
        return f"Email failed to send due to {response.text}"

send_test_html_email("Test Email", "<h1>Hello, World!</h1><p>This is a test email.</p>")

'Email failed to send due to {"id":"328120bc-4d7f-4efa-947f-a0225f761b3d"}'

### Did you receive the test email

If you get a 202, then you're good to go!

#### Certificate error

If you get an error SSL: CERTIFICATE_VERIFY_FAILED then students Chris S and Oleksandr K have suggestions:  
First run this: `!uv pip install --upgrade certifi`  
Next, run this:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

#### Other errors or no email

If there are other problems, you'll need to check your API key and your verified sender email address in the SendGrid dashboard

Or use the alternative implementation using "Resend Email" in community_contributions/2_lab2_with_resend_email

(Or - you could always replace the email sending code below with a Pushover call, or something to simply write to a flat file)

## Step 1: Agent workflow

In [7]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [8]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

In [9]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Reduce SOC 2 workload with AI-driven compliance support

Hi {{FirstName}},  

I’m reaching out from ComplAI. We help teams prepare for SOC 2 and stay audit-ready by using AI to streamline evidence collection, control mapping, and compliance workflows—so you spend less time chasing documentation and more time improving security.

If you’re currently in planning or preparing for an upcoming SOC 2 audit (or looking to reduce the effort between audits), I can share how teams typically use ComplAI to:  
- Map controls to your policies, systems, and evidence automatically  
- Generate audit-ready documentation faster  
- Maintain ongoing “ready state” with fewer manual checklists  

Would it be worth a quick 15-minute call next week to see whether this could help your team?  

Best regards,  
{{YourName}}  
{{Title}} | ComplAI  
{{Website}} | {{Phone}}

OPENAI_API_KEY is not set, skipping trace export


[Exporter] Export trace_id=trace_4ad3cb8422a544e78b612f6b6cc1983f, name=Agent workflow
[Exporter] Export span: {'object': 'trace.span', 'id': 'span_319eab54fa55467db1181c25', 'trace_id': 'trace_4ad3cb8422a544e78b612f6b6cc1983f', 'parent_id': 'span_65e7fd33e3e046208da8426a', 'started_at': '2026-06-13T23:24:35.310851+00:00', 'ended_at': '2026-06-13T23:24:37.249023+00:00', 'span_data': {'type': 'generation', 'input': [{'content': 'You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', 'role': 'system'}, {'role': 'user', 'content': 'Write a cold sales email'}], 'output': [{'id': '__fake_id__', 'created_at': 1781393076.164614, 'error': None, 'incomplete_details': None, 'instructions': None, 'metadata': None, 'model': 'gpt-5.4-nano', 'object': 'response', 'output': [{'id': '__fake_id__', 'content': [{'annotations': [], 'text': 'Subject: Reduce SOC 2 wor

OPENAI_API_KEY is not set, skipping trace export


[Exporter] Export trace_id=trace_18b1c2e01b7749f9a916192b9af29c59, name=Parallel cold emails


...

[Exporter] Export span: {'object': 'trace.span', 'id': 'span_5c457c90726c44d6877c9d3b', 'trace_id': 'trace_4c23f53a17c04ea0bcce720689dd0e38', 'parent_id': 'span_9e9775797f654de19febf157', 'started_at': '2026-06-13T23:35:25.988557+00:00', 'ended_at': '2026-06-13T23:35:28.762719+00:00', 'span_data': {'type': 'generation', 'input': [{'content': 'You are a busy sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write concise, to the point cold emails.', 'role': 'system'}, {'role': 'user', 'content': "Draft a high-converting cold email from ComplAI to a CEO. Start with 'Dear CEO'. Focus on outcomes (reduce compliance risk/cost, automate reviews, etc.)—use placeholders where needed. End with a single CTA question and include a short unsubscribe-friendly line if appropriate."}], 'output': [{'content': 'Subject: Cut SOC 2 prep time + risk—without adding headcount\n\nDear CEO,  \n\nSOC 2 audits shouldn’t

OPENAI_API_KEY is not set, skipping trace export


[Exporter] Export span: {'object': 'trace.span', 'id': 'span_e14ed83eb0a24543a771cc3d', 'trace_id': 'trace_4c23f53a17c04ea0bcce720689dd0e38', 'parent_id': 'span_6eba258f3311464a95f07431', 'started_at': '2026-06-13T23:35:29.479434+00:00', 'ended_at': '2026-06-13T23:35:32.564660+00:00', 'span_data': {'type': 'generation', 'input': [{'content': '\nYou are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.\n\nFollow these steps carefully:\n1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.\n\n2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.\n\n3. Use the send_email tool to send the best email (and only the best email) to the user.\n\nCrucial Rules:\n- You must use the sales agent tools to generate the drafts — do not write them yourself.\n- You must send ONE email

OPENAI_API_KEY is not set, skipping trace export


[Exporter] Export trace_id=trace_25fb7fb4e46a427fb22d36e5e1370d4c, name=Automated SDR
[Exporter] Export span: {'object': 'trace.span', 'id': 'span_f684fb20c144473e9d25bfc2', 'trace_id': 'trace_25fb7fb4e46a427fb22d36e5e1370d4c', 'parent_id': None, 'started_at': '2026-06-13T23:37:46.033726+00:00', 'ended_at': '2026-06-13T23:37:46.040446+00:00', 'span_data': {'type': 'agent', 'name': 'Sales Manager', 'handoffs': ['Email Manager'], 'tools': ['sales_agent1', 'sales_agent2', 'sales_agent3'], 'output_type': 'str'}, 'error': None}


In [10]:
message = "Write a cold sales email"

with trace("Parallel cold emails"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: SOC 2 readiness with AI help from ComplAI

Hi {{FirstName}},  

I’m reaching out from **ComplAI**—we help teams **prepare for SOC 2** and **stay audit-ready** using an AI-driven workflow. Instead of stitching together policies, evidence requests, and spreadsheets, ComplAI guides you through control implementation, evidence collection, and auditor-ready documentation.

**How teams typically use ComplAI:**
- Map controls to your current processes and systems  
- Generate and standardize audit evidence faster  
- Centralize documentation and track gaps toward SOC 2 readiness  
- Reduce manual back-and-forth during assessments  

If you’re currently planning for an upcoming audit (or tightening your SOC 2 posture), I’d like to ask: **what framework are you working toward—SOC 2 Type I or Type II—and what stage are you in today?**

Would you be open to a **10–15 minute** call next week?  

Best regards,  
{{YourName}}  
{{Title}} | ComplAI  
{{Phone}} | {{Website}}  
P.S. If you sha

In [11]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

In [12]:
message = "Write a cold sales email"

with trace("Selection from sales people"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
    outputs = [result.final_output for result in results]

    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

    best = await Runner.run(sales_picker, emails)

    print(f"Best sales email:\n{best.final_output}")


Best sales email:
Subject: Speed up your SOC 2 readiness with AI-assisted compliance (ComplAI)

Hi {{FirstName}},  

Quick question—are you currently preparing for SOC 2 (or responding to an audit), and finding that evidence collection and questionnaire work takes longer than expected?

At **ComplAI**, we use AI to help teams **prepare for SOC 2** by streamlining the work around compliance tasks and audit readiness—so you can spend less time hunting for evidence and more time demonstrating control effectiveness.

If you’re open to it, I’d like to share how teams typically use ComplAI to:
- Organize and generate SOC 2 evidence faster  
- Reduce manual, repetitive compliance workflows  
- Prepare for audits with less scramble and fewer gaps  

Would you be open to a **10–15 minute call next week**? If so, what does your schedule look like on **{{Day1}}** or **{{Day2}}?

Best regards,  
{{YourName}}  
{{Title}} | ComplAI  
{{Phone}} | {{Website}}


Now go and check out the trace:

https://platform.openai.com/traces

## Part 2: use of tools

Now we will add a tool to the mix.

Remember all that json boilerplate and the `handle_tool_calls()` function with the if logic..

In [13]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name)
)

In [14]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x109eb5a30>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

## Steps 2 and 3: Tools and Agent interactions

Remember all that boilerplate json?

Simply wrap your function with the decorator `@function_tool`

In [15]:
@function_tool
def send_html_email(subject: str, html_body: str) -> str:
    """Send out an email with the given subject and HTML body to all sales prospects using Resend"""
    
    from_email = "ed@edwarddonner.com"
    to_email = "ed.donner@gmail.com"
    RESEND_API_KEY = os.environ.get("RESEND_API_KEY")
    
    headers = {
        "Authorization": f"Bearer {RESEND_API_KEY}",
        "Content-Type": "application/json"
    }
    
    payload = {
        "from": f"Ed Donner <{from_email}>",
        "to": [to_email],
        "subject": subject,
        "html": html_body
    }
    
    response = requests.post("https://api.resend.com/emails", json=payload, headers=headers)
    
    if response.status_code == 202:
        return "Email sent successfully"
    else:
        return f"Email failed to send due to {response.text}"

### This has automatically been converted into a tool, with the boilerplate json created

In [19]:
# Let's look at it
send_html_email

FunctionTool(name='send_html_email', description='Send out an email with the given subject and HTML body to all sales prospects using Resend', params_json_schema={'properties': {'subject': {'title': 'Subject', 'type': 'string'}, 'html_body': {'title': 'Html Body', 'type': 'string'}}, 'required': ['subject', 'html_body'], 'title': 'send_html_email_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x113443b00>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### And you can also convert an Agent into a tool

In [20]:
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11409b560>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None)

### So now we can gather all the tools together:

A tool for each of our 3 email-writing agents

And a tool for our function to send emails

In [21]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3, send_html_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11409bce0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11409bf60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='sales_agent3', description='Write 

## And now it's time for our Sales Manager - our planning agent

In [22]:
# Improved instructions thanks to student Guillermo F.

instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", 
                      instructions=instructions, 
                      tools=tools, 
                      model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name))

message = "Send a cold sales email addressed to 'Dear CEO'"

with trace("Sales manager"):
    result = await Runner.run(sales_manager, message)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Wait - you didn't get an email??</h2>
            <span style="color:#ff7800;">With much thanks to student Chris S. for describing his issue and fixes. 
            If you don't receive an email after running the prior cell, here are some things to check: <br/>
            First, check your Spam folder! Several students have missed that the emails arrived in Spam!<br/>Second, print(result) and see if you are receiving errors about SSL. 
            If you're receiving SSL errors, then please check out theses <a href="https://chatgpt.com/share/680620ec-3b30-8012-8c26-ca86693d0e3d">networking tips</a> and see the note in the next cell. Also look at the trace in OpenAI, and investigate on the SendGrid website, to hunt for clues. Let me know if I can help!
            </span>
        </td>
    </tr>
</table>

### And one more suggestion to send emails from student Oleksandr on Windows 11:

If you are getting certificate SSL errors, then:  
Run this in a terminal: `uv pip install --upgrade certifi`

Then run this code:
```python
import certifi
import os
os.environ['SSL_CERT_FILE'] = certifi.where()
```

Thank you Oleksandr!

## Remember to check the trace

https://platform.openai.com/traces

And then check your email!!


### Handoffs represent a way an agent can delegate to an agent, passing control to it

Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across



In [24]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer",
                       instructions=subject_instructions, 
                       model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name))
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", 
                       instructions=html_instructions, 
                       model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name))
html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
# @function_tool
# def send_html_email(subject: str, html_body: str) -> Dict[str, str]:
#     """ Send out an email with the given subject and HTML body to all sales prospects """
#     sg = sendgrid.SendGridAPIClient(api_key=os.environ.get('SENDGRID_API_KEY'))
#     from_email = Email("ed@edwarddonner.com")  # Change to your verified sender
#     to_email = To("ed.donner@gmail.com")  # Change to your recipient
#     content = Content("text/html", html_body)
#     mail = Mail(from_email, to_email, subject, content).get()
#     sg.client.mail.send.post(request_body=mail)
#     return {"status": "success"}

In [25]:
tools = [subject_tool, html_tool, send_html_email]

In [26]:
tools

[FunctionTool(name='subject_writer', description='Write a subject for a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'subject_writer_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x114174680>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 FunctionTool(name='html_converter', description='Convert a text email body to an HTML email body', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'html_converter_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x114174180>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None),
 Function

In [27]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name),
    handoff_description="Convert an email to HTML and send it")


### Now we have 3 tools and 1 handoff

In [28]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent1_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11409bce0>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'sales_agent2_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function function_tool.<locals>._create_function_tool.<locals>._on_invoke_tool at 0x11409bf60>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None), FunctionTool(name='sales_agent3', description='Write a 

In [ ]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=OpenAIChatCompletionsModel(openai_client=azure_client, model=deployment_name))

message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Automated SDR"):
    result = await Runner.run(sales_manager, message)

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

### Remember to check the trace

https://platform.openai.com/traces

And then check your email!!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Can you identify the Agentic design patterns that were used here?<br/>
            What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?<br/>
            Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.<br/><br/>
            HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
            Then have the SDR respond to keep the conversation going! This may require some "vibe coding" 😂
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">This is immediately applicable to Sales Automation; but more generally this could be applied to  end-to-end automation of any business process through conversations and tools. Think of ways you could apply an Agent solution
            like this in your day job.
            </span>
        </td>
    </tr>
</table>

## Extra note:

Google has released their Agent Development Kit (ADK). It's not yet got the traction of the other frameworks on this course, but it's getting some attention. It's interesting to note that it looks quite similar to OpenAI Agents SDK. To give you a preview, here's a peak at sample code from ADK:

```
root_agent = Agent(
    name="weather_time_agent",
    model="gemini-2.0-flash",
    description="Agent to answer questions about the time and weather in a city.",
    instruction="You are a helpful agent who can answer user questions about the time and weather in a city.",
    tools=[get_weather, get_current_time]
)
```

Well, that looks familiar!

And a student has contributed a customer care agent in community_contributions that uses ADK.